In [1]:
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import joblib

# add path to project root
import sys
PROJECT_ROOT = Path.cwd()
sys.path.append(str(PROJECT_ROOT))
if PROJECT_ROOT.name == "notebooks":
    sys.path.append(str(PROJECT_ROOT.parent))
    
    
from src.preprocess.gaming_text_preprocessor import GamingTextPreprocessor
from src.preprocess.gaming_text_labeler import GamingTextLabeler
from src.ensemble import ensemble

# instantiate preprocessor and labeler
# gaming_preprocessor = GamingTextPreprocessor()
gaming_labeler = GamingTextLabeler()



In [2]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

repo_id = "jforward/bert-toxicity"
subfolder = "dota_binary_model"

tokenizer = AutoTokenizer.from_pretrained(repo_id, subfolder=subfolder)
model = AutoModelForSequenceClassification.from_pretrained(repo_id, subfolder=subfolder)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 10762.45it/s]


In [3]:
# config
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data' / 'processed_data').exists() and PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

CONFIG = {
    'seed': 7524,
    'data_dir': PROJECT_ROOT / 'data' / 'processed_data',
    'model_dir': PROJECT_ROOT / 'models',
    'text_col': 'message',
    'label_col': 'label'
}

np.random.seed(CONFIG['seed'])
seed = CONFIG['seed']
tc, lc = CONFIG['text_col'], CONFIG['label_col']

print('CONFIG loaded. Text column:', tc)

CONFIG loaded. Text column: message


In [4]:
# load WOT train data
d = CONFIG['data_dir']

wot_train = pd.read_parquet(d / 'wot' / 'x_train.parquet')
wot_train["data_source"] = "wot"


# load DOTA training data
dota_train = pd.read_parquet(d / 'dota' / 'x_train.parquet')
dota_train["data_source"] = "dota"

# combine datasets
train = pd.concat([wot_train, dota_train], ignore_index=True)

# clean labels and create 3-class label column
train = gaming_labeler.convert_three_class(
    train, 
    label_column=lc, 
    data_source_column='data_source',
    output_column='label_3class'
)

In [6]:
# tokenize training text
encoded_train = tokenizer(
    train[tc].fillna("").astype(str).tolist(),
    truncation=True,
    padding="max_length",
    max_length=128,
    return_tensors=None,
)

train["input_ids"] = encoded_train["input_ids"]
train["attention_mask"] = encoded_train["attention_mask"]